<a href="https://colab.research.google.com/github/NazmulHudaNabil/virtual-try-on-system/blob/main/Virtual_Try_On_System_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Install Dependencies

In [1]:
!pip install -q diffusers transformers accelerate torch torchvision \
               Pillow gradio numpy opencv-python-headless xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.0 MB/s eta 0:00:00


## Cell 2 — Imports

In [2]:
import numpy as np
import gradio as gr
import torch
from PIL import Image, ImageFilter
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from diffusers import StableDiffusionInpaintPipeline

# Use GPU if available, otherwise CPU (GPU strongly recommended)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = (512, 512)

print(f"✅ Running on: {DEVICE.upper()}")
if DEVICE == "cpu":
    print("⚠️  CPU detected — inpainting will be slow (~3–5 min). "
          "Switch to GPU: Runtime → Change runtime type → T4 GPU")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Running on: CUDA


## Cell 3 — Load Models

- **CLIPSeg** — zero-shot segmentation using a text prompt  
- **Stable Diffusion Inpainting** — fills the masked region with AI-generated clothing  

Both download automatically from Hugging Face Hub (free, one-time ~2 GB download).

In [3]:
# ── Load CLIPSeg (segmentation) ───────────────────────────────────────────────
print("Loading CLIPSeg segmentation model...")
seg_processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
seg_model     = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")
seg_model.eval()
print("✅ CLIPSeg ready.")

# ── Load Stable Diffusion Inpainting (runs locally — no API key needed) ───────
print("\nLoading Stable Diffusion Inpainting model (~2 GB, one-time download)...")
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
inpaint_pipe = inpaint_pipe.to(DEVICE)

# Memory optimization for Colab GPU
if DEVICE == "cuda":
    inpaint_pipe.enable_attention_slicing()

print("✅ Stable Diffusion Inpainting ready.")
print("\n🎉 All models loaded — ready to run!")

Loading CLIPSeg segmentation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


✅ Stable Diffusion Inpainting ready.

🎉 All models loaded — ready to run!


## Cell 4 — Pipeline Functions

In [4]:
# ── STEP A: Preprocess ────────────────────────────────────────────────────────

def preprocess_image(pil_image: Image.Image) -> Image.Image:
    """Resize and normalise to 512×512 RGB — required by Stable Diffusion."""
    return pil_image.convert("RGB").resize(IMAGE_SIZE, Image.LANCZOS)


# ── STEP B: Segment ───────────────────────────────────────────────────────────

def segment_clothing(image: Image.Image, body_part: str) -> Image.Image:
    """
    Use CLIPSeg to produce a binary mask for the selected clothing region.
    White pixels = area to inpaint (change clothing).
    """
    prompt_map = {
        "Upper Body": "upper clothing, shirt, top, jacket, blouse, sweater",
        "Lower Body": "lower clothing, pants, jeans, skirt, trousers, shorts",
    }
    prompt = prompt_map[body_part]

    # Run CLIPSeg
    inputs = seg_processor(
        text=[prompt], images=[image], return_tensors="pt", padding=True
    )
    with torch.no_grad():
        outputs = seg_model(**inputs)

    # Convert logits → probability map → binary mask
    logits     = outputs.logits.squeeze()                    # (H, W)
    probs      = torch.sigmoid(logits).cpu().numpy()         # 0–1
    prob_img   = Image.fromarray((probs * 255).astype(np.uint8))
    prob_img   = prob_img.resize(IMAGE_SIZE, Image.LANCZOS)

    # Threshold → binary, then blur edges for natural blending
    binary = (np.array(prob_img) > 100).astype(np.uint8) * 255
    mask   = Image.fromarray(binary).filter(ImageFilter.GaussianBlur(radius=4))

    return mask


# ── STEP C: Inpaint ───────────────────────────────────────────────────────────

def inpaint_clothing(image: Image.Image, mask: Image.Image, prompt: str) -> Image.Image:
    """
    Run Stable Diffusion Inpainting locally.
    Fills the white mask region with clothing described by the prompt.
    """
    full_prompt = (
        f"person wearing {prompt}, "
        "photorealistic, high quality, natural lighting, fashion photography"
    )
    negative_prompt = (
        "blurry, deformed, ugly, extra limbs, bad anatomy, "
        "watermark, text, low quality, cartoon"
    )

    result = inpaint_pipe(
        prompt          = full_prompt,
        negative_prompt = negative_prompt,
        image           = image,
        mask_image      = mask,
        num_inference_steps = 30,
        guidance_scale      = 7.5,
        strength            = 0.85,
    ).images[0]

    return result


print("✅ Pipeline functions defined.")

✅ Pipeline functions defined.


## Cell 5 — Main Orchestrator

In [5]:
def run_virtual_tryon(input_image, body_part, clothing_prompt):
    """
    Full end-to-end pipeline:
      1. Validate inputs
      2. Preprocess image → 512×512 RGB
      3. CLIPSeg → binary clothing mask
      4. SD Inpainting → new clothing applied
      5. Return result image + mask preview + status
    """
    if input_image is None:
        return None, None, "❌ Please upload an image."
    if not clothing_prompt.strip():
        return None, None, "❌ Please describe the clothing you want."

    try:
        print("[1/3] Preprocessing image...")
        image = preprocess_image(input_image)

        print(f"[2/3] Segmenting {body_part}...")
        mask = segment_clothing(image, body_part)

        print(f"[3/3] Inpainting with SD — prompt: '{clothing_prompt}'...")
        result = inpaint_clothing(image, mask, clothing_prompt)

        status = f"✅ Done! Applied '{clothing_prompt}' to {body_part.lower()}."
        print(status)
        return result, mask.convert("RGB"), status

    except Exception as e:
        msg = f"❌ Error: {str(e)}"
        print(msg)
        return None, None, msg


print("✅ Orchestrator ready.")

✅ Orchestrator ready.


## Cell 6 — Gradio UI

In [ ]:
css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Mono&display=swap');
* { font-family: 'DM Sans', sans-serif !important; }
body, .gradio-container { background: #0f0f11 !important; color: #e8e8ea !important; }
#app-header { text-align:center; padding:40px 0 24px; border-bottom:1px solid #1e1e24; margin-bottom:32px; }
#app-header h1 { font-size:2.4rem; font-weight:600; letter-spacing:-0.03em; color:#ffffff; margin:0 0 8px; }
#app-header p  { color:#6b6b78; font-size:0.95rem; font-weight:300; margin:0; }
.label-wrap    { font-size:0.78rem !important; font-weight:500 !important; letter-spacing:0.08em !important; text-transform:uppercase !important; color:#6b6b78 !important; }
.gr-button-primary  { background:#ffffff !important; color:#0f0f11 !important; font-weight:600 !important; border:none !important; border-radius:8px !important; padding:14px 28px !important; transition:opacity 0.15s !important; }
.gr-button-primary:hover { opacity:0.85 !important; }
.gr-button-secondary { background:transparent !important; border:1px solid #2a2a32 !important; color:#8888a0 !important; border-radius:8px !important; }
.gr-input, .gr-textarea { background:#16161c !important; border:1px solid #1e1e28 !important; border-radius:8px !important; color:#e8e8ea !important; }
.gr-box, .gr-panel { background:#13131a !important; border:1px solid #1e1e28 !important; border-radius:12px !important; }
#status-box textarea { font-family:'DM Mono',monospace !important; font-size:0.82rem !important; color:#a0a0b8 !important; background:#0d0d12 !important; }
"""

EXAMPLES = [
    ["black leather jacket"],
    ["white formal shirt"],
    ["red hoodie"],
    ["blue denim jeans"],
    ["floral summer top"],
    ["grey slim fit trousers"],
]

with gr.Blocks(css=css, title="Virtual Try-On") as demo:

    gr.HTML("""
    <div id="app-header">
        <h1>Virtual Try-On</h1>
        <p>Upload a photo · Choose a region · Describe the clothing · See the result</p>
    </div>""")

    with gr.Row(equal_height=False):

        # ── LEFT: Inputs ──────────────────────────────────────────────────────
        with gr.Column(scale=1):
            input_image = gr.Image(
                label="Your Photo", type="pil", height=320
            )
            body_part = gr.Radio(
                choices=["Upper Body", "Lower Body"],
                value="Upper Body",
                label="Region to Change",
            )
            clothing_prompt = gr.Textbox(
                label="Clothing Description",
                placeholder="e.g. black leather jacket, blue jeans, red hoodie…",
                lines=2,
            )
            gr.Examples(
                examples=EXAMPLES,
                inputs=[clothing_prompt],
                label="Quick picks",
            )
            with gr.Row():
                gr.ClearButton(
                    components=[input_image, clothing_prompt],
                    value="Clear", variant="secondary"
                )
                run_btn = gr.Button("✦ Try On", variant="primary")

        # ── RIGHT: Outputs ────────────────────────────────────────────────────
        with gr.Column(scale=1):
            output_image = gr.Image(
                label="Result", height=320, interactive=False
            )
            mask_preview = gr.Image(
                label="Segmentation Mask (detected region)",
                height=150, interactive=False
            )
            status_box = gr.Textbox(
                label="Pipeline Status",
                interactive=False,
                elem_id="status-box",
                lines=2,
            )

    run_btn.click(
        fn=run_virtual_tryon,
        inputs=[input_image, body_part, clothing_prompt],
        outputs=[output_image, mask_preview, status_box],
    )

    gr.HTML("""
    <div style="text-align:center;padding:32px 0 16px;color:#3a3a48;font-size:0.78rem;letter-spacing:0.05em;">
        CLIPSeg Segmentation &nbsp;·&nbsp; Stable Diffusion Inpainting (local) &nbsp;·&nbsp; Gradio UI
    </div>""")

demo.launch(share=True, debug=True)

/tmp/ipykernel_4147/716435248.py:26: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="Virtual Try-On") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3ffeff43a7e4e8b0a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 📖 Approach Summary

### Why no API key?
Both models run **locally inside Colab** — downloaded once from Hugging Face Hub for free, then executed on Colab's GPU. Zero cost, zero credit limits.

### Full Pipeline
```
Upload Image
  → Resize to 512×512 RGB
  → CLIPSeg → binary mask of clothing region
  → Stable Diffusion Inpainting (image + mask + prompt)
  → Show result image + mask preview
```

